# 🎯 OptiFull — Entrenamiento YOLOv8
**Sistema de Computer Vision para Tiendas FULL**

Este notebook entrena un modelo YOLOv8 para detección de personas y productos.
Ejecutá las celdas en orden de arriba hacia abajo.

## 1️⃣ Verificar GPU disponible

In [ ]:
import torch

print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print('⚠️ No se detectó GPU. Andá a Settings > Accelerator > GPU T4 x2')

## 2️⃣ Instalar dependencias

In [ ]:
!pip install ultralytics roboflow --quiet
from ultralytics import YOLO
import yaml, os
print('✅ Dependencias instaladas')

## 3️⃣ Descargar dataset desde Roboflow

Para obtener tu API key:
1. Entrás a roboflow.com
2. Settings → Roboflow API → copiás la key

In [ ]:
from roboflow import Roboflow

# ⚠️ REEMPLAZÁ estos valores con los de tu proyecto en Roboflow
API_KEY       = "TU_API_KEY_ACA"
WORKSPACE     = "nombre-de-tu-workspace"   # lo ves en la URL de roboflow.com
PROJECT_NAME  = "optifull"                 # nombre del proyecto que crearon
VERSION       = 1                          # versión del dataset

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
dataset = project.version(VERSION).download("yolov8")

print(f"✅ Dataset descargado en: {dataset.location}")

## 4️⃣ Verificar estructura del dataset

In [ ]:
import os

dataset_path = dataset.location

# Contar imágenes por split
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(dataset_path, split, 'images')
    if os.path.exists(split_path):
        n = len(os.listdir(split_path))
        print(f"{split:10}: {n} imágenes")

# Mostrar clases
yaml_path = os.path.join(dataset_path, 'data.yaml')
with open(yaml_path) as f:
    data_info = yaml.safe_load(f)

print(f"\nClases ({data_info['nc']} total):")
for i, cls in enumerate(data_info['names']):
    print(f"  {i}: {cls}")

## 5️⃣ Entrenar el modelo

Usamos `yolov8n.pt` (nano) para empezar rápido. Opciones disponibles:
- `yolov8n.pt` → más rápido, menos preciso (recomendado para empezar)
- `yolov8s.pt` → balance velocidad/precisión
- `yolov8m.pt` → más preciso, más lento

In [ ]:
model = YOLO('yolov8n.pt')  # descarga el modelo base preentrenado

results = model.train(
    data=yaml_path,
    epochs=100,          # cantidad de pasadas completas por el dataset
    imgsz=640,           # tamaño de imagen (estándar YOLO)
    batch=16,            # imágenes por batch (bajá a 8 si da error de memoria)
    patience=20,         # para si no mejora en 20 epochs (early stopping)
    save=True,           # guarda checkpoints
    project='optifull',  # carpeta donde guarda los resultados
    name='v1',           # nombre del experimento
    device=0,            # usa la GPU (cambiá a 'cpu' si no hay GPU)
    verbose=True
)

print('✅ Entrenamiento terminado!')

## 6️⃣ Ver resultados del entrenamiento

In [ ]:
from IPython.display import Image, display
import glob

# Mostrar curvas de entrenamiento
results_img = glob.glob('optifull/v1/results.png')
if results_img:
    display(Image(results_img[0]))

# Mostrar métricas finales
print("\n📊 Métricas finales:")
print(f"  mAP@50:     {results.results_dict.get('metrics/mAP50(B)', 'N/A'):.3f}")
print(f"  Precision:  {results.results_dict.get('metrics/precision(B)', 'N/A'):.3f}")
print(f"  Recall:     {results.results_dict.get('metrics/recall(B)', 'N/A'):.3f}")

# Ubicación del mejor modelo
best_model = 'optifull/v1/weights/best.pt'
print(f"\n💾 Mejor modelo guardado en: {best_model}")

## 7️⃣ Validar el modelo

In [ ]:
# Cargamos el mejor modelo y lo validamos en el set de validación
best_model_path = 'optifull/v1/weights/best.pt'
model_best = YOLO(best_model_path)

val_results = model_best.val(data=yaml_path)

print("\n✅ Validación completada")
print(f"mAP@50: {val_results.box.map50:.3f} (objetivo: >0.75)")

## 8️⃣ Probar con una imagen de prueba

In [ ]:
import glob
from IPython.display import Image, display

# Tomamos una imagen del set de validación para probar
test_images = glob.glob(os.path.join(dataset_path, 'valid/images/*.jpg'))[:3]

for img_path in test_images:
    results_pred = model_best.predict(
        source=img_path,
        conf=0.25,        # confianza mínima para mostrar una detección
        save=True,
        project='optifull',
        name='test_predictions'
    )

# Mostrar resultados
pred_images = glob.glob('optifull/test_predictions/*.jpg')
for img in pred_images[:3]:
    display(Image(img, width=640))

## 9️⃣ Descargar el modelo entrenado

Descargá el archivo `best.pt` para usarlo en tu proyecto local.

In [ ]:
# Opción A: Descargar directamente desde Kaggle
# (click derecho sobre el archivo en el panel de archivos → Download)

# Opción B: Subir automáticamente a Google Drive
# from google.colab import drive  # esto es solo para Colab, no Kaggle

# Opción C: Ver dónde está el archivo
import os
best_path = os.path.abspath('optifull/v1/weights/best.pt')
size_mb = os.path.getsize(best_path) / 1e6

print(f"📦 Modelo listo para descargar")
print(f"   Ruta: {best_path}")
print(f"   Tamaño: {size_mb:.1f} MB")
print("\n👉 En Kaggle: panel derecho → Output → optifull/v1/weights/best.pt → Download")

## 🔟 (Opcional) Extraer frames de videos para el dataset

Si necesitás extraer frames de los videos del disco externo antes de subirlos a Roboflow.

In [ ]:
# ESTO LO CORRÉS EN TU PC LOCAL, NO EN KAGGLE
# Lo incluimos acá como referencia

import cv2
from pathlib import Path

def extraer_frames(video_path, output_dir, cada_n_segundos=5, camara_id="cam1"):
    """
    Extrae frames de un video y los guarda como imágenes.
    
    video_path       : ruta al archivo de video (puede estar en el disco externo)
    output_dir       : carpeta donde guardar los frames
    cada_n_segundos  : frecuencia de extracción
    camara_id        : identificador de la cámara para el nombre del archivo
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cada_n_frames = int(fps * cada_n_segundos)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video: {Path(video_path).name}")
    print(f"FPS: {fps:.1f} | Total frames: {total_frames} | Extrayendo 1 cada {cada_n_segundos}s")
    
    i, guardados = 0, 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if i % cada_n_frames == 0:
            nombre = f"{camara_id}_frame_{guardados:05d}.jpg"
            cv2.imwrite(os.path.join(output_dir, nombre), frame)
            guardados += 1
        i += 1
    
    cap.release()
    print(f"✅ {guardados} frames guardados en {output_dir}")
    return guardados


# EJEMPLO DE USO (correr en tu PC local):
# extraer_frames(
#     video_path="E:/videos/camara1_semana1.mp4",
#     output_dir="./frames_dataset",
#     cada_n_segundos=5,
#     camara_id="cam1"
# )